# Module 10: Interactive Asyncio & Modern Structured Concurrency

Welcome to the deep-dive interactive lab on Python asyncio! In this notebook, you will explore:
1. **The Cooperative Event Loop:** Why single-threaded async beats OS threads for I/O bound workloads.
2. **The Blocking Trap:** Watching `time.sleep()` completely freeze other concurrent users, and how `asyncio.sleep()` yields control.
3. **Python 3.11+ Structured Concurrency:** Using `asyncio.TaskGroup` to eliminate task leakage.
4. **Rate Limiting & Queues:** Throttling concurrency with `asyncio.Semaphore` and `asyncio.Queue`.
5. **Bridging Legacy Sync Code:** Offloading blocking calls cleanly with `asyncio.to_thread`.
6. **Interactive Debugging:** Diagnosing and handling timeout cascades.

## 1. Cooperative Multitasking vs Preemptive Multithreading

In standard multithreading, the OS kernel forcefully interrupts threads (preemption). In `asyncio`, tasks run on a **single thread** and must voluntarily surrender control to the event loop at `await` points.

In [ ]:
import asyncio
import time


async def cooperative_task(name: str, delay: float):
    print(f'[{time.strftime("%X")}] Task {name}: Starting (yielding for {delay}s)')
    await asyncio.sleep(delay)  # <--- Cooperative yield to event loop!
    print(f'[{time.strftime("%X")}] Task {name}: Resumed and finished!')
    return f'{name}_DONE'

# In Jupyter, the event loop is already running, so we can await directly!
await cooperative_task('Alpha', 0.1)


## 2. The Blocking Trap: Why time.sleep() Freezes Servers

Watch what happens when an async endpoint accidentally calls a synchronous blocking function like `time.sleep()` or a non-async HTTP library. Because there is only ONE thread, the entire event loop is paralyzed.

In [ ]:
async def good_citizen(task_id: int):
    await asyncio.sleep(0.1)
    return f'Citizen_{task_id}'

async def bad_citizen():
    # SIMULATING A BLOCKING DATABASE CALL OR REQUESTS.GET()
    time.sleep(0.3)  # <--- BLOCKS THE ENTIRE PROCESS!
    return 'BAD_BLOCKING_RESULT'

t0 = time.perf_counter()
# When bad_citizen runs, it monopolizes the CPU
results = await asyncio.gather(bad_citizen(), good_citizen(1), good_citizen(2))
elapsed = time.perf_counter() - t0
print(f'Total elapsed time with blocking call: {elapsed:.3f}s (All tasks were delayed!)')


## 3. The Fix: Offloading Blocking I/O with asyncio.to_thread

Python 3.9+ introduced `asyncio.to_thread()`, which pushes blocking calls onto an underlying ThreadPoolExecutor without stalling the main event loop.

In [ ]:
def blocking_legacy_io():
    time.sleep(0.2)
    return 'Legacy_IO_Success'

t0 = time.perf_counter()
# By wrapping in to_thread, good_citizen can make progress concurrently!
results = await asyncio.gather(
    asyncio.to_thread(blocking_legacy_io),
    good_citizen(1),
    good_citizen(2)
)
elapsed = time.perf_counter() - t0
print(f'Elapsed time using asyncio.to_thread: {elapsed:.3f}s')
print(f'Results: {results}')


## 4. Modern Structured Concurrency with asyncio.TaskGroup (Python 3.11+)

`asyncio.gather()` had a notorious flaw: if one task raised an exception, the other tasks continued running as orphan zombie background coroutines. `TaskGroup` guarantees that when exiting the `async with` block, **all** spawned tasks have completed or been safely cancelled.

In [ ]:
async def subtask(num: int, delay: float):
    await asyncio.sleep(delay)
    return num * 10

results = []
async with asyncio.TaskGroup() as tg:
    # Spawn 3 concurrent tasks inside the group
    t1 = tg.create_task(subtask(1, 0.1))
    t2 = tg.create_task(subtask(2, 0.05))
    t3 = tg.create_task(subtask(3, 0.08))

# Outside the block, all tasks are guaranteed to have terminated
results = [t1.result(), t2.result(), t3.result()]
print(f'TaskGroup results: {results}')


## 5. Rate Limiting with asyncio.Semaphore

Spawning 10,000 coroutines simultaneously will overwhelm upstream servers and exhaust OS socket file descriptors. `asyncio.Semaphore` restricts active concurrent operations to a safe budget.

In [ ]:
sem = asyncio.Semaphore(2)  # Allow maximum 2 concurrent requests

async def rate_limited_fetch(item_id: int):
    async with sem:
        print(f'[Sem Acquired] Fetching item {item_id}...')
        await asyncio.sleep(0.1)
        print(f'[Sem Released] Item {item_id} fetched.')
        return item_id

# Launch 6 tasks; observe them executing in strict pairs of 2!
await asyncio.gather(*(rate_limited_fetch(i) for i in range(1, 7)))


## 6. Producer-Consumer Pipelines with asyncio.Queue

Decoupling ingestion from processing using asynchronous queues with backpressure and graceful worker termination.

In [ ]:
queue = asyncio.Queue(maxsize=5)

async def producer(n_items: int):
    for i in range(n_items):
        await queue.put(f'Event_{i}')
        print(f'Producer added: Event_{i}')
        await asyncio.sleep(0.02)
    # Put sentinel None to signal consumer termination
    await queue.put(None)

async def consumer():
    processed = []
    while True:
        item = await queue.get()
        if item is None:
            queue.task_done()
            break
        processed.append(item.lower())
        queue.task_done()
    return processed

consumer_task = asyncio.create_task(consumer())
await producer(5)
result = await consumer_task
print(f'Queue pipeline processed items: {result}')


## 7. Interactive Debugging Challenge: Resolving Timeout Cascades

**The Scenario:** A critical payment or auth service occasionally hangs. We must guard it with Python 3.11+ `asyncio.timeout()` and return a graceful fallback.

In [ ]:
async def slow_service():
    await asyncio.sleep(1.0)  # Simulating a slow upstream service
    return 'SLOW_DATA'

async def fetch_with_safe_timeout():
    try:
        # Python 3.11+ asyncio.timeout context manager
        async with asyncio.timeout(0.15):
            return await slow_service()
    except TimeoutError:
        print('[TIMEOUT HANDLED] Upstream service timed out. Returning cached fallback.')
        return 'FALLBACK_DATA'

safe_result = await fetch_with_safe_timeout()
print(f'Final Outcome: {safe_result}')


### 🎓 Key Takeaways for Production Mastery
1. **Never block the event loop:** Always use `asyncio.sleep()` or offload to threads via `asyncio.to_thread()`.
2. **Default to TaskGroup:** Favor `asyncio.TaskGroup` over `asyncio.gather` in Python 3.11+ for clean exception boundaries.
3. **Protect resources:** Always bound concurrent fanout with `asyncio.Semaphore`.
4. **Set deadlines:** Guard network calls with `asyncio.timeout()`.